# Qwen-Omni Inference for Cryptocurrency Video Analysis

This notebook demonstrates inference using the **Qwen2-Audio** (Qwen-Omni) model for cryptocurrency sentiment analysis.

## What is Qwen-Omni?

Qwen2-Audio (Qwen-Omni) is a **true multimodal model** that:
- Processes **audio and text simultaneously** in a single forward pass
- Uses **cross-attention** between audio and text modalities
- Can understand audio content (speech, tone, emotion) directly without transcription
- Generates text responses based on audio input

## Use Case

Test the base Qwen-Omni model on cryptocurrency TikTok videos before fine-tuning with LoRA.

## Architecture

```
┌─────────────────┐
│  Video Input    │
└────────┬────────┘
         │
    ┌────┴────┐
    │         │
[Audio]   [Frames]
    │         │
    │         └──────────────┐
    │                        │
    ▼                        ▼
┌────────────────┐    ┌──────────────┐
│  Qwen2-Audio   │    │ Future: Add  │
│  (Qwen-Omni)   │    │ Visual LoRA  │
│                │    │ (Optional)   │
│ Audio Encoder  │    └──────────────┘
│      +         │
│ Text Decoder   │
└───────┬────────┘
        │
        ▼
┌──────────────────┐
│ Sentiment Score  │
│ + Reasoning      │
└──────────────────┘
```

## Step 1: Setup and Installation

In [1]:
# Install required packages
!pip install -q transformers>=4.37.0 accelerate
!pip install -q librosa soundfile
!pip install -q torch torchvision torchaudio

print("✓ All packages installed!")

✓ All packages installed!


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
import subprocess
import re
from typing import Dict, List, Tuple

warnings.filterwarnings('ignore')

# Check CUDA
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Create directories
os.makedirs("./test_videos", exist_ok=True)
os.makedirs("./results", exist_ok=True)
os.makedirs("./temp", exist_ok=True)

print("\n✓ Setup complete!")

PyTorch Version: 2.7.1+cu118
CUDA Available: True
CUDA Device: NVIDIA H100 80GB HBM3
GPU Memory: 79.18 GB

✓ Setup complete!


## Step 2: Load Qwen2-Audio (Qwen-Omni) Model

Loading the base model before LoRA fine-tuning.

In [3]:
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

print("="*80)
print("LOADING QWEN2.5-OMNI MODEL")
print("="*80)

# Model selection - choose based on your GPU memory
# Options:
# - "Qwen/Qwen2.5-Omni-7B" (7B params, ~14GB GPU memory) - CORRECT MODEL
# - "Qwen/Qwen2-Audio-7B-Instruct" (older version)
# - "Qwen/Qwen2-Audio-7B-Instruct-Int4" (4-bit quantized, ~4GB GPU memory)

model_name = "Qwen/Qwen2.5-Omni-7B"
# model_name = "Qwen/Qwen2-Audio-7B-Instruct-Int4"  # Use this for lower memory

print(f"\nLoading model: {model_name}")
print("This may take a few minutes...\n")

# Load processor
processor = AutoProcessor.from_pretrained(model_name)
print("✓ Processor loaded")

# Load model
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Use bfloat16 for efficiency
    device_map="auto",           # Automatically distribute across GPUs
    trust_remote_code=True
)
print(f"✓ Model loaded on: {model.device}")

print("\n" + "="*80)
print("MODEL LOADED SUCCESSFULLY!")
print("="*80)

LOADING QWEN2.5-OMNI MODEL

Loading model: Qwen/Qwen2.5-Omni-7B
This may take a few minutes...



The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
You are using a model of type qwen2_5_omni to instantiate a model of type qwen2_audio. This is not supported for all configurations of models and can yield errors.
You are using a model of type qwen2_5_omni to instantiate a model of type qwen2_audio. This is not supported for all configurations of models and can yield errors.


✓ Processor loaded


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen2AudioForConditionalGeneration were not initialized from the model checkpoint at Qwen/Qwen2.5-Omni-7B and are newly initialized: ['audio_tower.conv1.bias', 'audio_tower.conv1.weight', 'audio_tower.conv2.bias', 'audio_tower.conv2.weight', 'audio_tower.embed_positions.weight', 'audio_tower.layer_norm.bias', 'audio_tower.layer_norm.weight', 'audio_tower.layers.0.fc1.bias', 'audio_tower.layers.0.fc1.weight', 'audio_tower.layers.0.fc2.bias', 'audio_tower.layers.0.fc2.weight', 'audio_tower.layers.0.final_layer_norm.bias', 'audio_tower.layers.0.final_layer_norm.weight', 'audio_tower.layers.0.self_attn.k_proj.weight', 'audio_tower.layers.0.self_attn.out_proj.bias', 'audio_tower.layers.0.self_attn.out_proj.weight', 'audio_tower.layers.0.self_attn.q_proj.bias', 'audio_tower.layers.0.self_attn.q_proj.weight', 'audio_tower.layers.0.self_attn.v_proj.bias', 'audio_tower.layers.0.self_attn.v_proj.weight', 'audio_tower.layers.0.self_attn_layer_norm.bias', 'audio_tower.layers.0.self

✓ Model loaded on: cuda:0

MODEL LOADED SUCCESSFULLY!


## Step 3: Qwen-Omni Inference Function

Qwen2.5-Omni processes video files directly (no audio extraction needed!)

In [ ]:
# No audio extraction needed! Qwen2.5-Omni processes video directly
print("✓ Qwen2.5-Omni handles video files natively (audio + visual combined)")

## Step 4: Qwen-Omni Video Analysis Function

This function performs sentiment analysis using video input directly (audio + visual).

In [4]:
def analyze_video_with_qwen_omni(video_path: str, video_name: str = "") -> Dict:
    """
    Analyze cryptocurrency sentiment from video using Qwen2.5-Omni.
    
    Args:
        video_path: Path to video file (.mp4)
        video_name: Name of video (for logging)
    
    Returns:
        Dict with sentiment_score, sentiment_class, confidence, reasoning
    """
    if not video_path or not os.path.exists(video_path):
        return {
            'sentiment_score': 0.0,
            'sentiment_class': 'NEUTRAL',
            'confidence': 'LOW',
            'reasoning': 'No video available'
        }
    
    # Create prompt for sentiment analysis
    prompt = """You are an expert financial analyst specializing in cryptocurrency sentiment analysis.

Watch this TikTok video about cryptocurrency/Dogecoin and analyze the overall sentiment.

Consider:
- **Bullish signals**: Words like "pump", "moon", "buy", "rocket", "gains", "profit", "surge", or excited/confident tone
- **Bearish signals**: Words like "dump", "crash", "sell", "loss", "drop", "falling", or worried/fearful tone
- **Neutral signals**: Words like "hold", "wait", "uncertain", "stable", or calm/analytical tone
- **Tone and emotion**: Excitement, fear, confidence, uncertainty
- **Visual cues**: Charts, graphs, emojis, gestures, expressions

Provide your analysis in this EXACT format:

SENTIMENT: [POSITIVE/NEGATIVE/NEUTRAL]
CONFIDENCE: [HIGH/MEDIUM/LOW]
SCORE: [number from -1.0 to +1.0, where -1.0 is extremely bearish, 0 is neutral, +1.0 is extremely bullish]
REASONING: [Brief explanation combining audio, visual, and tone analysis]
"""
    
    # Prepare conversation with video input
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "video", "video_url": video_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Process with Qwen2.5-Omni
    text = processor.apply_chat_template(
        conversation, 
        add_generation_prompt=True, 
        tokenize=False
    )
    
    # Qwen2.5-Omni processes video directly (audio + visual combined)
    inputs = processor(
        text=text,
        videos=video_path,
        return_tensors="pt",
        padding=True
    )
    inputs = inputs.to(model.device)
    
    # Generate response
    print(f"   → Analyzing video: {video_name}")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.0  # Deterministic for consistency
        )
    
    # Trim input tokens
    generated_ids = [
        output_ids[len(input_ids):] 
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    response = processor.batch_decode(
        generated_ids, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    # Parse response
    result = parse_sentiment_response(response)
    
    print(f"   ✓ Sentiment: {result['sentiment_class']} (Score: {result['sentiment_score']:.3f})")
    
    return result


def parse_sentiment_response(response: str) -> Dict:
    """
    Parse Qwen-Omni response to extract structured sentiment information.
    """
    sentiment_class = 'NEUTRAL'
    confidence = 'MEDIUM'
    score = 0.0
    reasoning = response
    
    # Extract sentiment class
    if 'SENTIMENT:' in response:
        match = re.search(r'SENTIMENT:\s*(\w+)', response, re.IGNORECASE)
        if match:
            sentiment_class = match.group(1).upper()
    
    # Extract confidence
    if 'CONFIDENCE:' in response:
        match = re.search(r'CONFIDENCE:\s*(\w+)', response, re.IGNORECASE)
        if match:
            confidence = match.group(1).upper()
    
    # Extract score
    if 'SCORE:' in response:
        match = re.search(r'SCORE:\s*([+-]?\d+\.?\d*)', response, re.IGNORECASE)
        if match:
            try:
                score = float(match.group(1))
                score = max(-1.0, min(1.0, score))
            except:
                pass
    
    # If no explicit score, infer from sentiment class and confidence
    if score == 0.0 and sentiment_class != 'NEUTRAL':
        confidence_map = {'HIGH': 0.8, 'MEDIUM': 0.5, 'LOW': 0.3}
        magnitude = confidence_map.get(confidence, 0.5)
        
        if 'POSITIVE' in sentiment_class:
            score = magnitude
        elif 'NEGATIVE' in sentiment_class:
            score = -magnitude
    
    # Extract reasoning
    if 'REASONING:' in response:
        match = re.search(r'REASONING:\s*(.+)', response, re.IGNORECASE | re.DOTALL)
        if match:
            reasoning = match.group(1).strip()
    
    return {
        'sentiment_score': score,
        'sentiment_class': sentiment_class,
        'confidence': confidence,
        'reasoning': reasoning,
        'raw_response': response
    }

print("✓ Qwen-Omni inference functions defined!")

✓ Qwen-Omni inference functions defined!


## Step 5: Test on Sample Videos

In [5]:
# List available videos
video_files = list(Path("./test_videos").glob("*.mp4"))
print(f"Found {len(video_files)} video files\n")

if len(video_files) > 0:
    # Show first 10 videos
    print("Sample videos:")
    for i, video in enumerate(video_files[:10], 1):
        print(f"  {i}. {video.name}")
    
    if len(video_files) > 10:
        print(f"  ... and {len(video_files) - 10} more")
else:
    print("⚠ No videos found in ./test_videos/ directory")
    print("Please add video files to ./test_videos/ folder")

Found 10 video files

Sample videos:
  1. 2025_02_07_1.mp4
  2. 2025_02_06_1.mp4
  3. 2025_02_07.mp4
  4. 2025_02_06_5.mp4
  5. 2025_02_04.mp4
  6. 2025_02_06.mp4
  7. 2025_02_06_4.mp4
  8. 2025_02_06_3.mp4
  9. 2025_02_04_1.mp4
  10. 2025_02_06_2.mp4


In [6]:
# Test on a single video
if len(video_files) > 0:
    test_video = video_files[0]  # Change index to test different videos
    
    print("="*80)
    print(f"TESTING ON: {test_video.name}")
    print("="*80)
    
    # Analyze video directly with Qwen-Omni
    print("\n1. Analyzing with Qwen-Omni (native video processing)...")
    result = analyze_video_with_qwen_omni(str(test_video), test_video.name)
    
    # Display results
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    print(f"\nSentiment Class: {result['sentiment_class']}")
    print(f"Sentiment Score: {result['sentiment_score']:.3f}")
    print(f"Confidence: {result['confidence']}")
    print(f"\nReasoning:\n{result['reasoning']}")
    
    print("\n" + "="*80)
    print("Raw Model Response:")
    print("="*80)
    print(result['raw_response'])
else:
    print("⚠ No videos available for testing")

TESTING ON: 2025_02_07_1.mp4

1. Analyzing with Qwen-Omni (native video processing)...


: 

## Step 6: Batch Processing (Optional)

Process multiple videos and save results.

In [ ]:
def process_videos_batch(
    video_dir: str = "./test_videos",
    output_file: str = "./results/qwen_omni_sentiment.csv",
    max_videos: int = None
) -> pd.DataFrame:
    """
    Process multiple videos and save sentiment analysis results.
    
    Args:
        video_dir: Directory containing video files
        output_file: Path to save results CSV
        max_videos: Maximum number of videos to process (None = all)
    
    Returns:
        DataFrame with results
    """
    video_files = list(Path(video_dir).glob("*.mp4"))
    
    if max_videos:
        video_files = video_files[:max_videos]
    
    print(f"\nProcessing {len(video_files)} videos...\n")
    
    results = []
    
    for i, video_path in enumerate(video_files, 1):
        print(f"\n[{i}/{len(video_files)}] {video_path.name}")
        print("-" * 60)
        
        try:
            # Analyze video directly (no audio extraction needed)
            result = analyze_video_with_qwen_omni(str(video_path), video_path.name)
            
            # Extract date from filename
            date_match = re.search(r'(\d{4})[_-](\d{2})[_-](\d{2})', video_path.name)
            date = f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}" if date_match else video_path.stem
            
            # Store result
            row = {
                'date': date,
                'video_path': video_path.name,
                'sentiment_score': result['sentiment_score'],
                'sentiment_class': result['sentiment_class'],
                'confidence': result['confidence'],
                'reasoning': result['reasoning'],
                'model': 'Qwen2.5-Omni-7B',
                'timestamp': datetime.now().isoformat()
            }
            results.append(row)
        
        except Exception as e:
            print(f"   ✗ Error: {str(e)}")
            continue
    
    # Save results
    if results:
        df = pd.DataFrame(results)
        df = df.sort_values('date')
        
        # Save detailed results
        df.to_csv(output_file, index=False)
        print(f"\n{'='*80}")
        print(f"✓ Results saved to: {output_file}")
        print(f"✓ Processed {len(results)} videos successfully")
        print(f"{'='*80}\n")
        
        # Display summary
        print("\nSentiment Distribution:")
        print(df['sentiment_class'].value_counts())
        print(f"\nAverage Sentiment Score: {df['sentiment_score'].mean():.3f}")
        
        return df
    else:
        print("\n⚠ No results to save")
        return pd.DataFrame()

print("✓ Batch processing function defined!")

In [ ]:
# Run batch processing on first 10 videos (for testing)
# Uncomment to process all videos: max_videos=None

results_df = process_videos_batch(
    video_dir="./test_videos",
    output_file="./results/qwen_omni_sentiment.csv",
    max_videos=10  # Change to None to process all videos
)

## Step 7: Visualize Results (Optional)

In [ ]:
import matplotlib.pyplot as plt

if len(results_df) > 0:
    # Create visualization
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Plot 1: Sentiment scores over time
    results_df['date_dt'] = pd.to_datetime(results_df['date'])
    results_df_sorted = results_df.sort_values('date_dt')
    
    axes[0].plot(results_df_sorted['date_dt'], results_df_sorted['sentiment_score'], 
                 marker='o', linewidth=2, markersize=6)
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Date', fontsize=11)
    axes[0].set_ylabel('Sentiment Score', fontsize=11)
    axes[0].set_title('Qwen-Omni: Sentiment Scores Over Time', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Plot 2: Sentiment distribution
    sentiment_counts = results_df['sentiment_class'].value_counts()
    colors_map = {'POSITIVE': 'green', 'NEGATIVE': 'red', 'NEUTRAL': 'gray'}
    colors = [colors_map.get(x, 'blue') for x in sentiment_counts.index]
    
    axes[1].bar(sentiment_counts.index, sentiment_counts.values, color=colors, alpha=0.7)
    axes[1].set_xlabel('Sentiment Class', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_title('Sentiment Distribution', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('./results/qwen_omni_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Visualization saved to: ./results/qwen_omni_visualization.png")
else:
    print("No results to visualize")

## Summary

This notebook demonstrates:

1. ✅ **Loading Qwen2-Audio (Qwen-Omni)** - Base model before LoRA fine-tuning
2. ✅ **Audio extraction** - From video files using ffmpeg
3. ✅ **Direct audio analysis** - Using Qwen-Omni's native audio understanding
4. ✅ **Structured output parsing** - Sentiment score, class, confidence, reasoning
5. ✅ **Batch processing** - Process multiple videos efficiently
6. ✅ **Results visualization** - Sentiment trends and distribution

## Next Steps for LoRA Fine-tuning:

1. **Prepare training data**:
   - Audio files from videos
   - Ground truth sentiment labels
   - Format: conversation pairs (audio + prompt → sentiment)

2. **Configure LoRA**:
   - Target modules: `q_proj`, `v_proj`, `k_proj`, `o_proj`
   - LoRA rank: 8-32 (typical)
   - LoRA alpha: 16-64

3. **Fine-tune**:
   - Use this inference pipeline to test before/after fine-tuning
   - Compare base model vs. fine-tuned model performance

4. **Evaluate**:
   - Measure accuracy, F1-score on validation set
   - Qualitative analysis of reasoning quality

---

**Model Used**: Qwen2-Audio-7B-Instruct (Base)

**Advantages**:
- Direct audio understanding (no transcription needed)
- Captures tone, emotion, emphasis
- End-to-end trainable with LoRA

**Ready for LoRA fine-tuning!** 🚀